# m2 — Assembly and Extraction

Per-sample: fastp QC → BWA-MEM/Minimap2 alignment → coverage gate → bcftools variant calling →
CDS extraction + HGVS amino-acid annotation.

| Step | Tool | Output |
|------|------|--------|
| 1 | fastp | Trimmed FASTQ, QC JSON/HTML |
| 2 | BWA-MEM / Minimap2 | Sorted BAM |
| 3 | samtools depth | Coverage QC gate (default ≥90% at ≥10×) |
| 4a–b | bcftools mpileup + call | VCF filtered to mmpR5 region |
| 4c | Biopython | CDS FASTA, protein FASTA, HGVS annotation TSV |

Outputs per sample saved to `Drive/…/output/{label}/`.

In [ ]:
# parameters
SAMPLE_CSV         = ""
SRR_LIST           = []
SRR_ACCESSION      = ""
DRIVE_OUTPUT       = "mmpR5_pipeline/output"
DRIVE_REF          = "mmpR5_pipeline/input"
READ_TYPE          = "illumina"
REF_SOURCE         = "drive"
REF_FILENAME       = "H37Rv_NC000962_MTBKO_20180411.fasta"
REF_NCBI_ACCESSION = "NC_000962.3"
REF_SEQ_NAME       = "H37Rv_NC000962_MTBKO_20180411"
RV0678_START       = 762572
RV0678_END         = 763123
MIN_DEPTH          = 10
COVERAGE_THRESHOLD = 0.90

In [ ]:
# CPU only — no GPU needed
# ── Load pipeline config (Drive JSON fallback) ──────────────────────────────
import json, shutil, subprocess, warnings, time, datetime, concurrent.futures
from pathlib import Path
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_CFG_PATH = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/pipeline_config.json")

def _load_config():
    if _CFG_PATH.exists():
        with open(_CFG_PATH) as _f:
            return json.load(_f)
    return {}

_cfg = _load_config()

def _p(key, default=None):
    """Resolve parameter: papermill-injected variable takes precedence over config JSON."""
    try:
        v = eval(key)                # injected by papermill
        return v if v is not None else _cfg.get(key, default)
    except Exception:
        return _cfg.get(key, default)

In [ ]:
# CPU only — no GPU needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE  = Path("/content/drive/MyDrive/ColabNotebooks")
OUTPUT_ROOT = DRIVE_BASE / _p("DRIVE_OUTPUT", "mmpR5_pipeline/output")
REF_DIR     = DRIVE_BASE / _p("DRIVE_REF",    "mmpR5_pipeline/input")
MODULES_DIR = DRIVE_BASE / "mmpR5_pipeline" / "modules"
print(f"Drive mounted. Output root: {OUTPUT_ROOT}")

In [ ]:
# CPU only — no GPU needed
# ── Resolve parameters ────────────────────────────────────────────────────────
_READ_TYPE          = _p("READ_TYPE", "illumina")
_REF_SOURCE         = _p("REF_SOURCE", "drive")
_REF_FILENAME       = _p("REF_FILENAME", "H37Rv_NC000962_MTBKO_20180411.fasta")
_REF_NCBI_ACCESSION = _p("REF_NCBI_ACCESSION", "NC_000962.3")
_REF_SEQ_NAME       = _p("REF_SEQ_NAME", "H37Rv_NC000962_MTBKO_20180411")
_RV0678_START       = int(_p("RV0678_START", 762572))
_RV0678_END         = int(_p("RV0678_END",   763123))
_MIN_DEPTH          = int(_p("MIN_DEPTH",          10))
_COV_THRESH         = float(_p("COVERAGE_THRESHOLD", 0.90))
_GENE_LENGTH        = _RV0678_END - _RV0678_START + 1  # 552 bp

# Resolve sample manifest from Drive (written by m1)
_manifest_csv = OUTPUT_ROOT / "sample_manifest.csv"
if _manifest_csv.exists():
    _mdf = pd.read_csv(str(_manifest_csv))
    _sample_manifest = _mdf.to_dict("records")
    for _sm in _sample_manifest:
        _sm["label"] = str(_sm.get("sample_label", _sm.get("srr", "")))
else:
    # Fallback: build from parameters
    _sample_manifest = []
    _sample_csv = _p("SAMPLE_CSV", "")
    _srr_list   = _p("SRR_LIST", [])
    _srr_single = _p("SRR_ACCESSION", "")
    if _sample_csv and Path(_sample_csv).exists():
        _df2 = pd.read_csv(_sample_csv)
        for _, _r in _df2.iterrows():
            _sample_manifest.append({"srr": str(_r["srr"]).strip(),
                "label": str(_r.get("sample_label", _r["srr"])).strip(),
                "phenotype": str(_r.get("phenotype", "U")).strip().upper()})
    elif _srr_list:
        _sample_manifest = [{"srr": s, "label": s, "phenotype": "U"} for s in _srr_list]
    elif _srr_single:
        _sample_manifest = [{"srr": _srr_single, "label": _srr_single, "phenotype": "U"}]
    else:
        raise ValueError("No samples. Run m1 first or set SAMPLE_CSV/SRR_ACCESSION.")

print(f"Samples to process: {len(_sample_manifest)}")

# ── Reference genome ──────────────────────────────────────────────────────────
_REF_WORK = Path("/content/reference"); _REF_WORK.mkdir(exist_ok=True)
REF_LOCAL = _REF_WORK / _REF_FILENAME
if REF_LOCAL.exists():
    print(f"Reference cached: {REF_LOCAL.name}")
elif _REF_SOURCE == "drive":
    shutil.copy2(REF_DIR / _REF_FILENAME, REF_LOCAL)
    print(f"Copied from Drive: {REF_LOCAL.name}")
elif _REF_SOURCE == "ncbi":
    !efetch -db nucleotide -id {_REF_NCBI_ACCESSION} -format fasta > {REF_LOCAL}
    _recs = list(SeqIO.parse(str(REF_LOCAL), "fasta"))
    _recs[0].id = _REF_SEQ_NAME; _recs[0].description = ""
    SeqIO.write(_recs, str(REF_LOCAL), "fasta")
    print(f"Downloaded from NCBI: {REF_LOCAL.name}")
_rec = next(SeqIO.parse(str(REF_LOCAL), "fasta"))
print(f"Reference: {_rec.id}  |  {len(_rec.seq):,} bp")
# ── Auto-detect chromosome name from FASTA header ─────────────────────────
if _rec.id != _REF_SEQ_NAME:
    print(f"[INFO] FASTA chrom '{_rec.id}' != param '{_REF_SEQ_NAME}' → using '{_rec.id}'")
    _REF_SEQ_NAME = _rec.id

In [ ]:
# CPU only — no GPU needed
# ── Step function definitions ─────────────────────────────────────────────────
_REF_INDEXED = False

_AA3 = {"A":"Ala","C":"Cys","D":"Asp","E":"Glu","F":"Phe","G":"Gly","H":"His",
        "I":"Ile","K":"Lys","L":"Leu","M":"Met","N":"Asn","P":"Pro","Q":"Gln",
        "R":"Arg","S":"Ser","T":"Thr","V":"Val","W":"Trp","Y":"Tyr","*":"Ter","X":"Xaa"}

def _fastp(label, r1, r2, wdir, odir):
    r1t = wdir / f"{label}_R1_trimmed.fastq.gz"
    r2t = None
    qcj = wdir / "fastp.json"
    cmd = ["fastp","-i",str(r1),"-o",str(r1t),
           "--json",str(qcj),"--html",str(wdir/"fastp.html"),"--thread","4"]
    if _READ_TYPE == "illumina" and r2:
        cmd += ["--detect_adapter_for_pe","-I",str(r2),"-O",str(wdir/f"{label}_R2_trimmed.fastq.gz")]
        r2t = wdir / f"{label}_R2_trimmed.fastq.gz"
    elif _READ_TYPE != "illumina":
        cmd += ["--disable_adapter_trimming","--length_required","200"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"fastp failed: {res.stderr[:300]}")
    with open(qcj) as fh:
        qc = json.load(fh)
    summ = qc.get("summary",{}).get("after_filtering",{})
    print(f"  [{label}] fastp: reads={summ.get('total_reads','?'):,}  Q30={summ.get('q30_rate',0)*100:.1f}%")
    for f in filter(None, [r1t, r2t, qcj, wdir/"fastp.html"]):
        if Path(f).exists(): shutil.copy2(f, odir/"01_qc"/Path(f).name)
    return r1t, r2t

def _align(label, r1t, r2t, wdir, odir):
    global _REF_INDEXED
    bam = wdir / f"{label}_sorted.bam"
    if not _REF_INDEXED and _READ_TYPE == "illumina":
        if not (_REF_WORK / (_REF_FILENAME+".bwt")).exists():
            subprocess.run(["bwa","index",str(REF_LOCAL)], check=True, capture_output=True)
            subprocess.run(["samtools","faidx",str(REF_LOCAL)], check=True)
        _REF_INDEXED = True
    rg = f"@RG\tID:{label}\tSM:{label}\tPL:{_READ_TYPE.upper()}"
    if _READ_TYPE == "illumina":
        cmd = ["bwa","mem","-R",rg,"-t","4",str(REF_LOCAL),str(r1t)]
        if r2t: cmd.append(str(r2t))
    elif _READ_TYPE == "nanopore":
        cmd = ["minimap2","-ax","map-ont","-t","4","-R",rg,str(REF_LOCAL),str(r1t)]
    else:
        cmd = ["minimap2","-ax","map-hifi","-t","4","-R",rg,str(REF_LOCAL),str(r1t)]
    aln  = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    sort = subprocess.Popen(["samtools","sort","-o",str(bam),"-"],
                            stdin=aln.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    _, _ = sort.communicate(); aln.wait()
    if aln.returncode != 0: raise RuntimeError("Alignment failed")
    subprocess.run(["samtools","index",str(bam)], check=True)
    cov = wdir / f"{label}_coverage_mmpR5.txt"
    with open(str(cov), "w") as fout:
        subprocess.run(["samtools","depth","-a",
            "-r",f"{_REF_SEQ_NAME}:{_RV0678_START}-{_RV0678_END}",str(bam)],
            stdout=fout, check=True)
    _cdf = pd.read_csv(str(cov), sep="\t", names=["chrom","pos","depth"])
    print(f"  [{label}] align: depth={_cdf['depth'].mean():.1f}x")
    flagstat = wdir / f"{label}_flagstat.txt"
    with open(str(flagstat),"w") as fout:
        subprocess.run(["samtools","flagstat",str(bam)], stdout=fout, check=True)
    for f in [bam, Path(str(bam)+".bai"), flagstat, cov]:
        shutil.copy2(f, odir/"02_alignment"/f.name)
    return bam

def _cov_qc(label, bam, wdir):
    cov = wdir / f"{label}_coverage_mmpR5.txt"
    if not cov.exists():
        with open(str(cov),"w") as fout:
            subprocess.run(["samtools","depth","-a",
                "-r",f"{_REF_SEQ_NAME}:{_RV0678_START}-{_RV0678_END}",str(bam)],
                stdout=fout, check=True)
    _cdf    = pd.read_csv(str(cov), sep="\t", names=["chrom","pos","depth"])
    covered = int((_cdf["depth"] >= _MIN_DEPTH).sum())
    frac    = covered / _GENE_LENGTH
    print(f"  [{label}] cov QC: {frac:.1%}  ", end="")
    if frac < _COV_THRESH:
        print(f"[FAIL — need {_COV_THRESH:.0%}]"); return False
    print("[PASS]"); return True

def _variants(label, bam, wdir, odir):
    vcf = wdir / f"{label}_variants.vcf.gz"
    mp  = subprocess.Popen(
        ["bcftools","mpileup","-f",str(REF_LOCAL),
         "--min-BQ","30","--annotate","FORMAT/AD,FORMAT/DP",str(bam)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    cl  = subprocess.Popen(
        ["bcftools","call","-mv","--ploidy","1","-Oz","-o",str(vcf)],
        stdin=mp.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    _, _ = cl.communicate(); mp.wait()
    subprocess.run(["bcftools","index",str(vcf)], check=True)
    mmpr5_vcf = wdir / f"{label}_mmpR5.vcf"
    with open(str(mmpr5_vcf),"w") as fout:
        subprocess.run(["bcftools","view",
            "-r",f"{_REF_SEQ_NAME}:{_RV0678_START}-{_RV0678_END}",str(vcf)],
            stdout=fout, check=True)
    variants = []
    with open(str(mmpr5_vcf)) as fh:
        for line in fh:
            if line.startswith("#"): continue
            p = line.strip().split("\t")
            if len(p) >= 5:
                variants.append({"pos":int(p[1]),"ref":p[3],"alt":p[4],"qual":p[5],"info":p[7] if len(p)>7 else ""})
    var_df = pd.DataFrame(variants) if variants else pd.DataFrame(columns=["pos","ref","alt","qual","info"])
    print(f"  [{label}] variants: {len(var_df)} in mmpR5 region")
    smry = wdir / f"{label}_mmpR5_summary.txt"
    with open(str(smry),"w") as fh:
        fh.write(f"Sample: {label}\nVariants: {len(var_df)}\n")
        fh.write(var_df[["pos","ref","alt","qual"]].to_string(index=False))
    vdir = odir / "03_variants"
    for f in [vcf, Path(str(vcf)+".csi"), mmpr5_vcf, smry]:
        if f.exists(): shutil.copy2(f, vdir/f.name)
    return var_df

def _cds_annotation(label, var_df, wdir, odir):
    nt_fa = wdir / f"{label}_mmpR5_cds.fasta"
    aa_fa = wdir / f"{label}_mmpR5_protein.fasta"
    aa_tsv= wdir / f"{label}_aa_annotation.tsv"
    ref_rec   = SeqIO.read(str(REF_LOCAL), "fasta")
    ref_bases = list(str(ref_rec.seq))
    wt_cds    = Seq("".join(ref_bases[_RV0678_START-1:_RV0678_END])).reverse_complement()
    for _, row in var_df.iterrows():
        p0 = int(row["pos"]) - 1
        if ref_bases[p0].upper() == row["ref"].upper():
            ref_bases[p0] = row["alt"]
    mmpr5_nt = Seq("".join(ref_bases))[_RV0678_START-1:_RV0678_END].reverse_complement()
    mmpr5_aa = str(mmpr5_nt[:-3].translate())
    aa_anns = []
    for _, row in var_df.iterrows():
        pos_g, ref_nt, alt_nt = int(row["pos"]), row["ref"], row["alt"]
        if len(ref_nt) != 1 or len(alt_nt) != 1:
            aa_anns.append({"genomic_pos":pos_g,"ref_nt":ref_nt,"alt_nt":alt_nt,
                "cds_pos":"?","codon_pos":"?","aa_pos":"?","wt_codon":"?","mut_codon":"?",
                "wt_aa":"?","mut_aa":"?","hgvs_c":f"c.indel@{pos_g}","hgvs_p":"p.indel","effect":"indel"})
            continue
        cds0 = _RV0678_END - pos_g; codon0 = cds0//3; pip = cds0%3; aa1 = codon0+1
        cs = codon0*3; wtc = str(wt_cds[cs:cs+3])
        mc_list = list(wtc); mc_list[pip] = str(Seq(alt_nt).complement()); mc = "".join(mc_list)
        wta = str(Seq(wtc).translate()); muta = str(Seq(mc).translate())
        hc = f"c.{cds0+1}{ref_nt.upper()}>{alt_nt.upper()}"
        hp = f"p.{_AA3.get(wta,wta)}{aa1}{_AA3.get(muta,muta)}"
        eff = "synonymous" if wta==muta else "nonsense" if muta=="*" else "missense"
        aa_anns.append({"genomic_pos":pos_g,"ref_nt":ref_nt,"alt_nt":alt_nt,
            "cds_pos":cds0+1,"codon_pos":pip+1,"aa_pos":aa1,
            "wt_codon":wtc,"mut_codon":mc,"wt_aa":wta,"mut_aa":muta,
            "hgvs_c":hc,"hgvs_p":hp,"effect":eff})
    aa_df = pd.DataFrame(aa_anns)
    mut_label = "wild-type"
    if not aa_df.empty:
        mut_label = ", ".join(
            row["hgvs_p"] if row["effect"] != "indel" else row["hgvs_c"]
            for _, row in aa_df.iterrows())
    SeqIO.write(SeqRecord(Seq(str(mmpr5_nt)),id=f"{label}_mmpR5_cds",description=f"Rv0678 CDS|{mut_label}"),
                str(nt_fa),"fasta")
    SeqIO.write(SeqRecord(Seq(mmpr5_aa),id=f"{label}_mmpR5_protein",description=f"Rv0678 prot|{mut_label}"),
                str(aa_fa),"fasta")
    aa_df.to_csv(str(aa_tsv), sep="\t", index=False)
    sdir = odir / "04_sequences"
    for f in [nt_fa, aa_fa, aa_tsv]:
        shutil.copy2(f, sdir/f.name)
    print(f"  [{label}] CDS: {mut_label}")
    return aa_df, mmpr5_aa, str(mmpr5_nt), mut_label

In [ ]:
# CPU only — no GPU needed
# ── Batch processing loop ─────────────────────────────────────────────────────
_m2_status = {}
print("=" * 70)
print("  M2 — ASSEMBLY AND EXTRACTION BATCH LOOP")
print("=" * 70)

for _sm in _sample_manifest:
    _srr, _label = _sm["srr"], _sm["label"]
    _wdir = Path(f"/content/{_label}")
    _odir = OUTPUT_ROOT / _label
    print(f"{'─'*65}")

    # Checkpointing: skip if AA TSV exists
    _aa_tsv = _wdir / f"{_label}_aa_annotation.tsv"
    if _aa_tsv.exists():
        print(f"[SKIP] {_label} — aa_annotation.tsv present")
        _m2_status[_label] = "SKIPPED"
        continue

    print(f"[RUN] {_label} ({_srr})")
    _wdir.mkdir(parents=True, exist_ok=True)
    for _s in ["01_qc","02_alignment","03_variants","04_sequences","05_structure","06_scores"]:
        (_odir/_s).mkdir(parents=True, exist_ok=True)

    try:
        # Locate FASTQ
        _r1p = _wdir/f"{_srr}_1.fastq"; _r2p = _wdir/f"{_srr}_2.fastq"
        _rsg = _wdir/f"{_srr}.fastq"
        if _r1p.exists() and _r2p.exists():   _r1, _r2 = _r1p, _r2p
        elif _r1p.exists():                    _r1, _r2 = _r1p, None
        elif _rsg.exists():                    _r1, _r2 = _rsg, None
        else:
            raise RuntimeError(f"FASTQ not found for {_srr}. Run m1 first.")
        _r1t, _r2t = _fastp(_label, _r1, _r2, _wdir, _odir)
        _bam        = _align(_label, _r1t, _r2t, _wdir, _odir)
        if not _cov_qc(_label, _bam, _wdir):
            _m2_status[_label] = "FAILED_COV_QC"; continue
        _var_df     = _variants(_label, _bam, _wdir, _odir)
        _cds_annotation(_label, _var_df, _wdir, _odir)
        _m2_status[_label] = "DONE"
        print(f"[DONE] {_label}")
    except Exception as _exc:
        import traceback; traceback.print_exc()
        _m2_status[_label] = f"FAILED: {_exc}"

# Save status
_st_df = pd.DataFrame([{"label":k,"m2_status":v} for k,v in _m2_status.items()])
_st_df.to_csv(str(OUTPUT_ROOT/"m2_status.csv"), index=False)
print("\nM2 complete. Status summary:")
print(_st_df.to_string(index=False))